## Portable path configuration
This release copy contains no personal or NAS paths. Set `GAO2026_DATA_ROOT` for deposited data and `GAO2026_RAW_ROOT` only when running raw-image stages (raw revision microscopy is not included).


In [ ]:
from pathlib import Path
import os
_default_repo_root = Path.cwd().resolve()
GAO2026_DATA_ROOT = Path(os.environ.get('GAO2026_DATA_ROOT', _default_repo_root / 'data')).expanduser().resolve()
GAO2026_RAW_ROOT = Path(os.environ.get('GAO2026_RAW_ROOT', GAO2026_DATA_ROOT / 'external_raw')).expanduser().resolve()
GAO2026_OUTPUT_ROOT = Path(os.environ.get('GAO2026_OUTPUT_ROOT', _default_repo_root / 'outputs')).expanduser().resolve()
GAO2026_EXTERNAL_ROOT = Path(os.environ.get('GAO2026_EXTERNAL_ROOT', GAO2026_DATA_ROOT / 'external')).expanduser().resolve()


# mTetR locus mobility analysis: all biological replicates

This self-contained notebook analyzes raw MCP/mTetR movies for
Usp5 and Dnmt3L, with two biological replicates per gene.

Dataset assignment:

- Usp5 rep1: 260618; Usp5 rep2: 260703
- Dnmt3L rep1: 260618; Dnmt3L rep2: 260619

The 260703 Usp5 dataset is assigned as rep2 because the student
OneNote rep2 figure reports Active n=39/Inactive n=28, exactly
matching the legacy classification table values ON n=39/OFF n=28.

Analysis rules are the corrected v2 rules: Gao-style MCP state
classification, actual mTetR frame-0 coordinate, Trackpy
`link_df(search_range=2, memory=4)`, and a fixed 0.03 s/frame,
66-lag MSD axis.

Reader-facing labels use Active/Inactive. Internal state values
remain ON/OFF for compatibility with the existing CSV and fit
tables.

Existing raw data and previous output folders are not modified.


In [ ]:
from __future__ import annotations

import json
import math
import os
import platform
import sys
import traceback
import warnings
import xml.etree.ElementTree as ET
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import nd2
import numpy as np
import pandas as pd
import scipy
import skimage
import tifffile
import trackpy as tp
from IPython.display import Image as IPythonImage
from IPython.display import display
from scipy.optimize import curve_fit
from skimage.feature import peak_local_max
from skimage.filters import gaussian
from skimage.restoration import rolling_ball

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 200

# Four biological-replicate datasets used for the publication figure.
# Override the NAS root when the mount name differs:
# MSD_KO_ROOT=/path/to/Ko MSD_ALL_REPLICATES_OUTPUT_DIR=/path/to/output ...
KO_ROOT = Path(
    os.environ.get(
        "MSD_KO_ROOT",
        str(GAO2026_RAW_ROOT / 'Public-TS-873-2/Microscope/Ko'),
    )
)
ALL_REPLICATES_OUTPUT_DIR = Path(
    os.environ.get(
        "MSD_ALL_REPLICATES_OUTPUT_DIR",
        str(KO_ROOT / "MSD_all_replicates_analysis_v3"),
    )
)
OUTPUT_TAG = os.environ.get(
    "MSD_ALL_REPLICATES_OUTPUT_TAG",
    "new_MSD_v3_all_replicates",
)
MAX_SAMPLES = int(os.environ.get("MSD_MAX_SAMPLES", "0"))
SAMPLE_IDS_FILTER = {
    value.strip()
    for value in os.environ.get("MSD_SAMPLE_IDS", "").split(",")
    if value.strip()
}

DATASET_CONFIGS = [
    {
        "dataset_id": "260618_Usp5",
        "gene": "Usp5",
        "replicate": 1,
        "data_dir": KO_ROOT
        / "260618-live-CFP-YFP_MSD"
        / "260618_Usp5"
        / "rawdata",
    },
    {
        "dataset_id": "260703_Usp5",
        "gene": "Usp5",
        "replicate": 2,
        "data_dir": KO_ROOT
        / "260703-live-CFP-YFP_MSD"
        / "260703_Usp5"
        / "rawdata",
    },
    {
        "dataset_id": "260618_Dnmt3L",
        "gene": "Dnmt3L",
        "replicate": 1,
        "data_dir": KO_ROOT
        / "260618-live-CFP-YFP_MSD"
        / "260618_Dnmt3L"
        / "rawdata",
    },
    {
        "dataset_id": "260619_Dnmt3L",
        "gene": "Dnmt3L",
        "replicate": 2,
        "data_dir": KO_ROOT
        / "260619-live-CFP-YFP_MSD"
        / "260619_Dnmt3L"
        / "rawdata",
    },
]

ALL_REPLICATES_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Image and channel definitions.
DEFAULT_PIXEL_SIZE_UM = 0.13
MCP_CHANNEL_INDEX = 0
MTETR_CHANNEL_INDEX = 1

# Gao snapshot criterion.
MCP_RATIO_THRESHOLD = 1.15
MCP_MAX_DISTANCE_PX = 3.0
MCP_RATIO_RADIUS_PX = 3
ROI_HALF_SIZE_PX = 9  # 19 x 19 ROI, as in Gao snapshot analysis.

# Detection and linking.
INITIAL_SEARCH_FRAMES = 5
TRACKPY_DIAMETER = 5
TRACKPY_TOPN = 1  # The paper specifies one candidate mTetR spot per ROI.
FIT_RADIUS_PX = 4
MIN_TRACK_SNR = 3.0
LINK_SEARCH_RANGE_PX = 2.0
LINK_MEMORY_FRAMES = 4
MIN_CONSECUTIVE_DETECTED_FRAMES = 51  # "more than 50" frames.

# Paper-like preprocessing.
GAUSSIAN_SIGMA_PX = 0.5
ROLLING_BALL_RADIUS_PX = 5

# Publication-reproduction MSD time axis.
ANALYSIS_FRAME_INTERVAL_S = 0.03
MAX_LAG_FRAMES = 66
PLOT_MAX_TIME_S = MAX_LAG_FRAMES * ANALYSIS_FRAME_INTERVAL_S
PLOT_Y_MAX_UM2 = 0.04

# Anomalous diffusion fit settings.
FIT_INITIAL_D_ALPHA = 0.005
FIT_INITIAL_ALPHA = 0.5
FIT_D_ALPHA_BOUNDS = (0.0, np.inf)
FIT_ALPHA_BOUNDS = (1e-6, 1.999999)
FIT_WEIGHTING = "unweighted"
FIT_CI_Z = 1.96

MANUAL_SEEDS: dict[str, tuple[float, float]] = {
    # "cell001": (x_px, y_px),
}

if str(tp.__version__) != "0.5.0":
    warnings.warn(
        f"trackpy {tp.__version__} is installed; the paper reports trackpy 0.5.0."
    )

expected_design = {
    (gene, replicate)
    for gene in ["Usp5", "Dnmt3L"]
    for replicate in [1, 2]
}
observed_design = {
    (dataset["gene"], int(dataset["replicate"]))
    for dataset in DATASET_CONFIGS
}
assert observed_design == expected_design
assert len({dataset["dataset_id"] for dataset in DATASET_CONFIGS}) == 4
for dataset in DATASET_CONFIGS:
    if not Path(dataset["data_dir"]).is_dir():
        raise FileNotFoundError(dataset["data_dir"])

print(f"Datasets: {len(DATASET_CONFIGS)}")
for dataset in DATASET_CONFIGS:
    print(
        f"- {dataset['gene']} rep{dataset['replicate']}: "
        f"{dataset['data_dir']}"
    )
print(f"Output root: {ALL_REPLICATES_OUTPUT_DIR}")
print(f"Per-dataset output tag: {OUTPUT_TAG}")
print(f"trackpy: {tp.__version__}")
print(
    "Analysis rules: "
    f"search_range={LINK_SEARCH_RANGE_PX}, "
    f"memory={LINK_MEMORY_FRAMES}, "
    f"dt={ANALYSIS_FRAME_INTERVAL_S:.2f} s, "
    f"max_lag={MAX_LAG_FRAMES} ({PLOT_MAX_TIME_S:.2f} s)"
)


## 1. Movie loading and ND2 timing audit


In [ ]:
def _first_defined(*values):
    for value in values:
        if value is not None:
            return value
    return None


def _safe_float(value) -> float | None:
    try:
        if value is None:
            return None
        if isinstance(value, (list, tuple, np.ndarray)) and len(value) == 1:
            value = value[0]
        value = float(value)
        return value if np.isfinite(value) else None
    except Exception:
        return None


def _parse_ome_metadata(ome_xml: str | None) -> dict[str, float]:
    if not ome_xml:
        return {}
    try:
        root = ET.fromstring(ome_xml)
        pixels = root.find(".//{*}Pixels")
        if pixels is None:
            return {}
        parsed: dict[str, float] = {}
        if "PhysicalSizeX" in pixels.attrib:
            parsed["pixel_size_um"] = float(pixels.attrib["PhysicalSizeX"])
        if "TimeIncrement" in pixels.attrib:
            parsed["metadata_mtetr_interval_s"] = float(
                pixels.attrib["TimeIncrement"]
            )
        return parsed
    except Exception:
        return {}


def _normalize_event_times_seconds(values) -> np.ndarray:
    arr = pd.to_numeric(pd.Series(list(values)), errors="coerce").to_numpy(dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size < 2:
        return np.array([], dtype=float)
    positive_diffs = np.diff(arr)
    positive_diffs = positive_diffs[
        np.isfinite(positive_diffs) & (positive_diffs > 0)
    ]
    if positive_diffs.size == 0:
        return np.array([], dtype=float)
    if float(np.median(positive_diffs)) > 10:
        arr = arr / 1000.0
    return arr


def _time_values_from_events(events) -> np.ndarray:
    if events is None:
        return np.array([], dtype=float)
    if isinstance(events, pd.DataFrame):
        time_columns = [
            column for column in events.columns if "time" in str(column).lower()
        ]
        for column in time_columns:
            arr = _normalize_event_times_seconds(events[column].to_numpy())
            if arr.size >= 2:
                return arr
    if isinstance(events, dict):
        for key, value in events.items():
            if "time" in str(key).lower():
                try:
                    arr = _normalize_event_times_seconds(value)
                    if arr.size >= 2:
                        return arr
                except Exception:
                    pass
            if isinstance(value, (dict, list, tuple, pd.DataFrame)):
                arr = _time_values_from_events(value)
                if arr.size >= 2:
                    return arr
    if isinstance(events, (list, tuple)) and events:
        if isinstance(events[0], dict):
            keys = sorted(
                {
                    key
                    for item in events
                    for key in item
                    if "time" in str(key).lower()
                }
            )
            for key in keys:
                arr = _normalize_event_times_seconds(
                    [item.get(key) for item in events if key in item]
                )
                if arr.size >= 2:
                    return arr
        else:
            return _normalize_event_times_seconds(events)
    return np.array([], dtype=float)


def _nd2_timing_metadata(nd2_file) -> dict[str, float]:
    timing: dict[str, float] = {}
    events = None
    for kwargs in ({}, {"orient": "records"}):
        try:
            events_method = getattr(nd2_file, "events", None)
            if callable(events_method):
                events = events_method(**kwargs)
                break
        except TypeError:
            continue
        except Exception:
            break

    times_s = _time_values_from_events(events)
    if times_s.size >= 2:
        timing["mcp_to_first_mtetr_gap_s"] = float(times_s[1] - times_s[0])
    if times_s.size >= 3:
        mtetr_diffs = np.diff(times_s[1:])
        mtetr_diffs = mtetr_diffs[
            np.isfinite(mtetr_diffs) & (mtetr_diffs > 0)
        ]
        if mtetr_diffs.size:
            timing["metadata_mtetr_interval_s"] = float(np.median(mtetr_diffs))

    if "metadata_mtetr_interval_s" not in timing:
        candidate_paths = [
            ("metadata", "contents", "frame_interval_ms"),
            ("metadata", "channels", 0, "time", "periodMs"),
            ("metadata", "time", "periodMs"),
        ]
        for parts in candidate_paths:
            current = nd2_file
            try:
                for part in parts:
                    if isinstance(part, int):
                        current = current[part]
                    else:
                        current = (
                            getattr(current, part)
                            if hasattr(current, part)
                            else current[part]
                        )
            except Exception:
                continue
            parsed = _safe_float(current)
            if parsed is not None and parsed > 0:
                timing["metadata_mtetr_interval_s"] = parsed / 1000.0
                break
    return timing


def load_tiff_movie(movie_path: str | Path) -> tuple[np.ndarray, dict[str, float]]:
    path = Path(movie_path)
    movie = np.asarray(tifffile.imread(path))
    metadata: dict[str, float] = {}
    with tifffile.TiffFile(path) as tif:
        imagej_meta = tif.imagej_metadata or {}
        if "finterval" in imagej_meta:
            parsed = _safe_float(imagej_meta["finterval"])
            if parsed is not None:
                metadata["metadata_mtetr_interval_s"] = parsed
        metadata.update(_parse_ome_metadata(tif.ome_metadata))
    return movie, metadata


def load_nd2_movie(movie_path: str | Path) -> tuple[np.ndarray, dict[str, float]]:
    path = Path(movie_path)
    movie = np.asarray(nd2.imread(path))
    metadata: dict[str, float] = {}
    with nd2.ND2File(path) as nd2_file:
        try:
            voxel = nd2_file.voxel_size()
            if getattr(voxel, "x", None) is not None:
                metadata["pixel_size_um"] = float(voxel.x)
        except Exception:
            pass
        metadata.update(_nd2_timing_metadata(nd2_file))
    return movie, metadata


def prepare_movie_arrays(
    raw_movie: np.ndarray,
    movie_path: str | Path,
) -> tuple[np.ndarray, np.ndarray, str]:
    arr = np.squeeze(np.asarray(raw_movie))
    if arr.ndim == 3:
        if arr.shape[0] < 2:
            raise ValueError(f"{movie_path}: fewer than two frames: {arr.shape}")
        return arr[0], arr[1:], "3D (time, y, x)"
    if arr.ndim != 4:
        raise ValueError(f"{movie_path}: unsupported shape {arr.shape}")
    if arr.shape[0] >= 10 and arr.shape[1] <= 4:
        return (
            arr[0, MCP_CHANNEL_INDEX],
            arr[1:, MTETR_CHANNEL_INDEX],
            "4D (time, channel, y, x)",
        )
    if arr.shape[0] <= 4 and arr.shape[1] >= 10:
        return (
            arr[MCP_CHANNEL_INDEX, 0],
            arr[MTETR_CHANNEL_INDEX, 1:],
            "4D (channel, time, y, x)",
        )
    raise ValueError(f"{movie_path}: could not interpret shape {arr.shape}")


def load_movie_file(
    sample: dict,
) -> tuple[np.ndarray, np.ndarray, dict[str, float], str]:
    movie_path = Path(sample["movie_path"])
    if movie_path.suffix.lower() == ".nd2":
        raw_movie, metadata = load_nd2_movie(movie_path)
    elif movie_path.suffix.lower() in {".tif", ".tiff"}:
        raw_movie, metadata = load_tiff_movie(movie_path)
    else:
        raise ValueError(f"Unsupported format: {movie_path}")
    mcp_image, mtetr_movie, structure = prepare_movie_arrays(raw_movie, movie_path)
    return mcp_image, mtetr_movie, metadata, structure


def resolve_pixel_size(sample: dict, metadata: dict[str, float]) -> tuple[float, str]:
    if sample.get("pixel_size_um") is not None:
        return float(sample["pixel_size_um"]), "sample_override"
    if metadata.get("pixel_size_um") is not None:
        return float(metadata["pixel_size_um"]), "file_metadata"
    return DEFAULT_PIXEL_SIZE_UM, "default_fallback"


print("Movie loading and timing-audit functions ready.")


## 2. Preprocessing, mTetR detection, and Trackpy linking


In [ ]:
def robust_sigma(values: np.ndarray) -> float:
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return 1e-6
    median = float(np.median(values))
    mad = float(np.median(np.abs(values - median)))
    sigma = 1.4826 * mad
    if sigma <= 0:
        sigma = float(np.std(values))
    return max(sigma, 1e-6)


def preprocess_frame(
    image: np.ndarray,
    gaussian_sigma_px: float = GAUSSIAN_SIGMA_PX,
    rolling_ball_radius_px: int = ROLLING_BALL_RADIUS_PX,
) -> np.ndarray:
    img = np.asarray(image, dtype=np.float32)
    blurred = gaussian(img, sigma=gaussian_sigma_px, preserve_range=True)
    background = rolling_ball(blurred, radius=rolling_ball_radius_px)
    result = blurred - background
    result[result < 0] = 0
    return result.astype(np.float32, copy=False)


def preprocess_movie(movie: np.ndarray) -> np.ndarray:
    return np.asarray([preprocess_frame(frame) for frame in movie], dtype=np.float32)


def extract_square_patch(
    image: np.ndarray,
    center_xy: tuple[float, float],
    half_size: int,
) -> tuple[np.ndarray, tuple[int, int]]:
    x_center, y_center = center_xy
    x0 = max(0, int(round(x_center)) - half_size)
    x1 = min(image.shape[1], int(round(x_center)) + half_size + 1)
    y0 = max(0, int(round(y_center)) - half_size)
    y1 = min(image.shape[0], int(round(y_center)) + half_size + 1)
    return image[y0:y1, x0:x1], (x0, y0)


def build_roi_bounds(
    center_xy: tuple[float, float],
    image_shape: tuple[int, int],
    half_size: int,
) -> tuple[int, int, int, int]:
    x_center, y_center = center_xy
    x0 = max(0, int(round(x_center)) - half_size)
    x1 = min(image_shape[1], int(round(x_center)) + half_size + 1)
    y0 = max(0, int(round(y_center)) - half_size)
    y1 = min(image_shape[0], int(round(y_center)) + half_size + 1)
    return x0, x1, y0, y1


def patch_border_values(patch: np.ndarray) -> np.ndarray:
    arrays = [patch[0, :], patch[-1, :]]
    if patch.shape[0] > 2:
        arrays.extend([patch[1:-1, 0], patch[1:-1, -1]])
    return np.concatenate([np.ravel(array) for array in arrays]).astype(
        np.float32, copy=False
    )


def gaussian_2d(
    coords,
    amplitude,
    x0,
    y0,
    sigma_x,
    sigma_y,
    background,
):
    xx, yy = coords
    exponent = -(
        ((xx - x0) ** 2) / (2.0 * sigma_x**2)
        + ((yy - y0) ** 2) / (2.0 * sigma_y**2)
    )
    return (background + amplitude * np.exp(exponent)).ravel()


def refine_center_gaussian(
    image: np.ndarray,
    approx_xy: tuple[float, float],
    fit_radius: int = FIT_RADIUS_PX,
) -> dict:
    patch, (x_origin, y_origin) = extract_square_patch(
        image, approx_xy, fit_radius
    )
    if patch.size == 0:
        return {
            "x_px": np.nan,
            "y_px": np.nan,
            "snr": np.nan,
            "fit_success": False,
            "fit_method": "empty_patch",
        }
    border = (
        patch_border_values(patch)
        if patch.shape[0] >= 2 and patch.shape[1] >= 2
        else patch.ravel()
    )
    background = float(np.median(border))
    sigma_bg = robust_sigma(border)
    yy, xx = np.indices(patch.shape, dtype=np.float32)
    shifted = patch.astype(np.float32) - background
    shifted[shifted < 0] = 0
    if shifted.sum() <= 0:
        y_peak, x_peak = np.unravel_index(np.argmax(patch), patch.shape)
        amplitude = float(np.max(patch) - background)
        return {
            "x_px": x_origin + float(x_peak),
            "y_px": y_origin + float(y_peak),
            "amplitude": amplitude,
            "background": background,
            "snr": amplitude / sigma_bg,
            "fit_success": False,
            "fit_method": "peak_only",
        }

    x_centroid = float((xx * shifted).sum() / shifted.sum())
    y_centroid = float((yy * shifted).sum() / shifted.sum())
    amplitude0 = float(np.max(patch) - background)
    try:
        params, _ = curve_fit(
            gaussian_2d,
            (xx, yy),
            patch.ravel(),
            p0=[
                max(amplitude0, 1.0),
                x_centroid,
                y_centroid,
                1.2,
                1.2,
                background,
            ],
            bounds=(
                [
                    0.0,
                    0.0,
                    0.0,
                    0.5,
                    0.5,
                    float(np.min(patch)),
                ],
                [
                    float(np.max(patch) * 3.0 + 1.0),
                    float(patch.shape[1] - 1),
                    float(patch.shape[0] - 1),
                    3.5,
                    3.5,
                    float(np.max(patch)),
                ],
            ),
            maxfev=10000,
        )
        amplitude, x_fit, y_fit, _, _, bg_fit = params
        return {
            "x_px": x_origin + float(x_fit),
            "y_px": y_origin + float(y_fit),
            "amplitude": float(amplitude),
            "background": float(bg_fit),
            "snr": float(amplitude) / sigma_bg,
            "fit_success": True,
            "fit_method": "gaussian_fit",
        }
    except Exception:
        return {
            "x_px": x_origin + x_centroid,
            "y_px": y_origin + y_centroid,
            "amplitude": amplitude0,
            "background": background,
            "snr": amplitude0 / sigma_bg,
            "fit_success": True,
            "fit_method": "centroid_fallback",
        }


def local_maximum(image: np.ndarray) -> tuple[float, float]:
    peaks = peak_local_max(
        image,
        min_distance=2,
        exclude_border=False,
        num_peaks=10,
    )
    if peaks.size == 0:
        y_peak, x_peak = np.unravel_index(np.argmax(image), image.shape)
        return float(x_peak), float(y_peak)
    values = [image[y, x] for y, x in peaks]
    y_peak, x_peak = peaks[int(np.argmax(values))]
    return float(x_peak), float(y_peak)


def detect_seed_from_mean(
    mtetr_movie_processed: np.ndarray,
    sample_id: str,
) -> dict:
    if sample_id in MANUAL_SEEDS:
        x_seed, y_seed = MANUAL_SEEDS[sample_id]
        return {
            "seed_x_px": float(x_seed),
            "seed_y_px": float(y_seed),
            "seed_mode": "manual",
            "seed_frames": 1,
        }
    n_frames = min(INITIAL_SEARCH_FRAMES, len(mtetr_movie_processed))
    mean_image = np.mean(mtetr_movie_processed[:n_frames], axis=0)
    candidates = tp.locate(
        mean_image,
        diameter=TRACKPY_DIAMETER,
        topn=TRACKPY_TOPN,
    )
    if candidates is not None and not candidates.empty:
        sort_column = (
            "mass"
            if "mass" in candidates.columns
            else "signal"
            if "signal" in candidates.columns
            else None
        )
        if sort_column is not None:
            candidates = candidates.sort_values(
                sort_column, ascending=False
            ).reset_index(drop=True)
        best = candidates.iloc[0]
        fit = refine_center_gaussian(
            mean_image, (float(best["x"]), float(best["y"]))
        )
        return {
            "seed_x_px": float(fit["x_px"]),
            "seed_y_px": float(fit["y_px"]),
            "seed_mode": "trackpy_mean_only",
            "seed_frames": int(n_frames),
        }
    x_peak, y_peak = local_maximum(mean_image)
    fit = refine_center_gaussian(mean_image, (x_peak, y_peak))
    return {
        "seed_x_px": float(fit["x_px"]),
        "seed_y_px": float(fit["y_px"]),
        "seed_mode": "localmax_mean_only",
        "seed_frames": int(n_frames),
    }


def detect_mtetr_candidates(
    mtetr_movie_processed: np.ndarray,
    seed_xy: tuple[float, float],
) -> tuple[pd.DataFrame, tuple[int, int, int, int]]:
    roi_bounds = build_roi_bounds(
        seed_xy,
        mtetr_movie_processed.shape[1:],
        ROI_HALF_SIZE_PX,
    )
    x0, x1, y0, y1 = roi_bounds
    records: list[dict] = []
    detection_id = 0

    for frame_index, frame in enumerate(mtetr_movie_processed):
        roi = frame[y0:y1, x0:x1]
        located = tp.locate(
            roi,
            diameter=TRACKPY_DIAMETER,
            topn=TRACKPY_TOPN,
        )
        if located is None or located.empty:
            x_peak, y_peak = local_maximum(roi)
            located = pd.DataFrame(
                [{"x": x_peak, "y": y_peak, "mass": np.nan, "signal": np.nan}]
            )

        for _, candidate in located.iterrows():
            approx_xy = (
                float(x0 + candidate["x"]),
                float(y0 + candidate["y"]),
            )
            fit = refine_center_gaussian(frame, approx_xy)
            if not (
                np.isfinite(fit["x_px"])
                and np.isfinite(fit["y_px"])
                and np.isfinite(fit["snr"])
            ):
                continue
            if float(fit["snr"]) < MIN_TRACK_SNR:
                continue
            records.append(
                {
                    "detection_id": detection_id,
                    "frame": int(frame_index),
                    "x": float(fit["x_px"]),
                    "y": float(fit["y_px"]),
                    "spot_snr": float(fit["snr"]),
                    "fit_method": str(fit["fit_method"]),
                    "locate_mass": _safe_float(candidate.get("mass")),
                    "locate_signal": _safe_float(candidate.get("signal")),
                }
            )
            detection_id += 1

    features = pd.DataFrame.from_records(records)
    if features.empty:
        raise RuntimeError("No mTetR candidates passed the SNR filter.")
    if not (features["frame"] == 0).any():
        raise RuntimeError(
            "No mTetR spot was detected in the first actual mTetR frame."
        )
    return features, roi_bounds


def link_mtetr_candidates(
    features: pd.DataFrame,
    seed_xy: tuple[float, float],
    n_frames: int,
    pixel_size_um: float,
) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    linked = tp.link_df(
        features.copy(),
        search_range=LINK_SEARCH_RANGE_PX,
        memory=LINK_MEMORY_FRAMES,
        pos_columns=["x", "y"],
        t_column="frame",
    )
    linked = linked.sort_values(["particle", "frame", "detection_id"]).reset_index(
        drop=True
    )

    first = linked[linked["frame"] == 0].copy()
    first["seed_distance_px"] = np.hypot(
        first["x"] - float(seed_xy[0]),
        first["y"] - float(seed_xy[1]),
    )
    first = first.sort_values(
        ["seed_distance_px", "spot_snr"],
        ascending=[True, False],
    )
    selected_first = first.iloc[0]
    selected_particle = int(selected_first["particle"])
    selected_detection_id = int(selected_first["detection_id"])

    selected = linked[linked["particle"] == selected_particle].copy()
    if selected["frame"].duplicated().any():
        selected = (
            selected.sort_values(["frame", "spot_snr"], ascending=[True, False])
            .drop_duplicates("frame", keep="first")
            .copy()
        )

    frame_table = pd.DataFrame({"frame": np.arange(n_frames, dtype=int)})
    selected = selected.rename(
        columns={
            "x": "x_px",
            "y": "y_px",
        }
    )
    track = frame_table.merge(
        selected[
            [
                "frame",
                "detection_id",
                "particle",
                "x_px",
                "y_px",
                "spot_snr",
                "fit_method",
                "locate_mass",
                "locate_signal",
            ]
        ],
        on="frame",
        how="left",
    )
    track["full_movie_frame"] = track["frame"] + 1
    track["time_s"] = track["frame"] * ANALYSIS_FRAME_INTERVAL_S
    track["tracking_status"] = np.where(
        track["x_px"].notna(),
        "linked_detection",
        "missing",
    )
    track["used_for_msd"] = track["x_px"].notna() & track["y_px"].notna()
    track["x_um"] = track["x_px"] * pixel_size_um
    track["y_um"] = track["y_px"] * pixel_size_um

    first_track = track.loc[track["frame"] == 0].iloc[0]
    first_x_px = float(first_track["x_px"])
    first_y_px = float(first_track["y_px"])
    track["dx_um"] = (track["x_px"] - first_x_px) * pixel_size_um
    track["dy_um"] = (track["y_px"] - first_y_px) * pixel_size_um
    track["disp_um"] = np.hypot(track["dx_um"], track["dy_um"])

    track["step_from_prev_detection_px"] = np.nan
    valid_indices = track.index[track["used_for_msd"]].to_numpy()
    if valid_indices.size >= 2:
        valid_xy = track.loc[valid_indices, ["x_px", "y_px"]].to_numpy(dtype=float)
        steps = np.hypot(
            np.diff(valid_xy[:, 0]),
            np.diff(valid_xy[:, 1]),
        )
        track.loc[valid_indices[1:], "step_from_prev_detection_px"] = steps

    selection = {
        "selected_particle": selected_particle,
        "selected_first_detection_id": selected_detection_id,
        "first_mtetr_x_px": first_x_px,
        "first_mtetr_y_px": first_y_px,
        "first_mtetr_snr": float(first_track["spot_snr"]),
    }
    return track, linked, selection


def longest_true_run(values: np.ndarray) -> int:
    maximum = 0
    current = 0
    for value in np.asarray(values, dtype=bool):
        if value:
            current += 1
            maximum = max(maximum, current)
        else:
            current = 0
    return int(maximum)


def trajectory_qc(track: pd.DataFrame) -> dict:
    detected = track["used_for_msd"].fillna(False).astype(bool).to_numpy()
    max_consecutive = longest_true_run(detected)
    valid_frames = int(detected.sum())
    missing_fraction = float(1.0 - detected.mean())
    median_snr = (
        float(np.nanmedian(track.loc[detected, "spot_snr"]))
        if detected.any()
        else np.nan
    )
    passed = max_consecutive >= MIN_CONSECUTIVE_DETECTED_FRAMES
    return {
        "qc_pass": bool(passed),
        "valid_frames": valid_frames,
        "missing_fraction": missing_fraction,
        "max_consecutive_detected_frames": max_consecutive,
        "median_track_snr": median_snr,
        "qc_rule": (
            f"max_consecutive_detected_frames >= "
            f"{MIN_CONSECUTIVE_DETECTED_FRAMES}"
        ),
    }


print("mTetR detection and trackpy linking functions ready.")


## 3. Gao-style Active/Inactive classification


In [ ]:
def compute_relative_intensity_ratio(
    image_roi: np.ndarray,
    center_xy_roi: tuple[float, float],
    radius_px: int = MCP_RATIO_RADIUS_PX,
) -> dict:
    image_roi = np.asarray(image_roi, dtype=np.float32)
    x_center, y_center = center_xy_roi
    pad = int(radius_px)
    padded = np.pad(image_roi, pad, mode="constant")
    coords = np.asarray(
        [[y_center + pad, x_center + pad]],
        dtype=np.float32,
    )
    refined = tp.refine_com(
        padded,
        padded,
        radius=pad,
        coords=coords,
    )
    mass_column = "raw_mass" if "raw_mass" in refined.columns else "mass"
    area = math.pi * pad**2
    signal_intensity = float(refined[mass_column].iloc[0]) / area
    roi_mean = float(np.mean(image_roi))
    ratio = signal_intensity / max(roi_mean, 1e-6)
    return {
        "relative_intensity": ratio,
        "signal_intensity": signal_intensity,
        "roi_mean_intensity": roi_mean,
        "method": "trackpy_refine_com_mass_over_area_over_roi_mean",
    }


def classify_mcp_state(
    mcp_image_raw: np.ndarray,
    mcp_image_processed: np.ndarray,
    first_mtetr_xy: tuple[float, float],
) -> dict:
    x_mtetr, y_mtetr = first_mtetr_xy
    roi_bounds = build_roi_bounds(
        first_mtetr_xy,
        mcp_image_raw.shape,
        ROI_HALF_SIZE_PX,
    )
    x0, x1, y0, y1 = roi_bounds
    raw_roi = np.asarray(mcp_image_raw[y0:y1, x0:x1], dtype=np.float32)
    processed_roi = np.asarray(
        mcp_image_processed[y0:y1, x0:x1],
        dtype=np.float32,
    )
    if raw_roi.size == 0:
        raise RuntimeError("MCP ROI is empty.")

    located = tp.locate(
        processed_roi,
        diameter=TRACKPY_DIAMETER,
        topn=1,
    )
    if located is None or located.empty:
        return {
            "state": "OFF",
            "MCP_status": "no_peak_detected",
            "mcp_x_px": np.nan,
            "mcp_y_px": np.nan,
            "distance_px": np.nan,
            "relative_intensity": np.nan,
            "signal_intensity": np.nan,
            "roi_mean_intensity": float(np.mean(raw_roi)),
            "intensity_method": "none",
            "roi_bounds": roi_bounds,
        }

    candidate = located.iloc[0]
    fit_roi = refine_center_gaussian(
        processed_roi,
        (float(candidate["x"]), float(candidate["y"])),
    )
    mcp_x_px = float(x0 + fit_roi["x_px"])
    mcp_y_px = float(y0 + fit_roi["y_px"])
    distance_px = float(
        np.hypot(mcp_x_px - x_mtetr, mcp_y_px - y_mtetr)
    )
    center_roi = (float(fit_roi["x_px"]), float(fit_roi["y_px"]))
    intensity = compute_relative_intensity_ratio(raw_roi, center_roi)
    ratio = float(intensity["relative_intensity"])
    active = (
        np.isfinite(ratio)
        and ratio >= MCP_RATIO_THRESHOLD
        and distance_px <= MCP_MAX_DISTANCE_PX
    )
    return {
        "state": "ON" if active else "OFF",
        "MCP_status": "focus_present" if active else "focus_absent",
        "mcp_x_px": mcp_x_px,
        "mcp_y_px": mcp_y_px,
        "distance_px": distance_px,
        "relative_intensity": ratio,
        "signal_intensity": intensity["signal_intensity"],
        "roi_mean_intensity": intensity["roi_mean_intensity"],
        "intensity_method": intensity["method"],
        "roi_bounds": roi_bounds,
    }


def mtetr_ratio_for_audit(
    mtetr_first_raw: np.ndarray,
    first_mtetr_xy: tuple[float, float],
) -> float:
    roi_bounds = build_roi_bounds(
        first_mtetr_xy,
        mtetr_first_raw.shape,
        ROI_HALF_SIZE_PX,
    )
    x0, x1, y0, y1 = roi_bounds
    raw_roi = np.asarray(mtetr_first_raw[y0:y1, x0:x1], dtype=np.float32)
    center_roi = (first_mtetr_xy[0] - x0, first_mtetr_xy[1] - y0)
    return float(
        compute_relative_intensity_ratio(raw_roi, center_roi)[
            "relative_intensity"
        ]
    )


print("Gao-style MCP classification functions ready.")


## 4. MSD, ensemble summary, anomalous fit, and plotting


In [ ]:
ACTIVITY_DISPLAY_LABELS = {
    "ON": "Active",
    "OFF": "Inactive",
    "EXCLUDE": "Excluded",
    "ERROR": "Error",
}


def activity_display_table(table: pd.DataFrame) -> pd.DataFrame:
    displayed = table.copy()
    for source_column, display_column in [
        ("state", "activity"),
        ("raw_state", "raw_activity"),
    ]:
        if source_column not in displayed.columns:
            continue
        original = displayed[source_column]
        displayed[source_column] = original.map(
            ACTIVITY_DISPLAY_LABELS
        ).fillna(original)
        displayed = displayed.rename(
            columns={source_column: display_column}
        )
    return displayed

def compute_time_averaged_msd(
    track: pd.DataFrame,
    sample_id: str,
    state: str,
) -> pd.DataFrame:
    coords = track[["x_um", "y_um"]].to_numpy(dtype=float)
    valid = np.isfinite(coords[:, 0]) & np.isfinite(coords[:, 1])
    records: list[dict] = []
    max_lag = min(MAX_LAG_FRAMES, len(track) - 1)
    for lag in range(1, max_lag + 1):
        valid_pairs = valid[:-lag] & valid[lag:]
        if not valid_pairs.any():
            continue
        deltas = coords[lag:] - coords[:-lag]
        squared_displacement = np.sum(deltas[valid_pairs] ** 2, axis=1)
        records.append(
            {
                "sample_id": sample_id,
                "state": state,
                "lag_frame": int(lag),
                "lag_time_s": float(lag * ANALYSIS_FRAME_INTERVAL_S),
                "msd_um2": float(np.mean(squared_displacement)),
                "n_pairs": int(valid_pairs.sum()),
            }
        )
    return pd.DataFrame.from_records(records)


def summarize_ensemble_msd(msd_per_trajectory: pd.DataFrame) -> pd.DataFrame:
    columns = [
        "state",
        "lag_frame",
        "lag_time_s",
        "n_trajectories",
        "mean_msd_um2",
        "sem_msd_um2",
        "ci95_low_um2",
        "ci95_high_um2",
    ]
    if msd_per_trajectory.empty:
        return pd.DataFrame(columns=columns)
    summary = (
        msd_per_trajectory.groupby(["state", "lag_frame"], as_index=False)
        .agg(
            lag_time_s=("lag_time_s", "mean"),
            n_trajectories=("sample_id", "nunique"),
            mean_msd_um2=("msd_um2", "mean"),
            std_msd_um2=("msd_um2", "std"),
        )
        .sort_values(["state", "lag_frame"])
    )
    summary["std_msd_um2"] = summary["std_msd_um2"].fillna(0.0)
    summary["sem_msd_um2"] = summary["std_msd_um2"] / np.sqrt(
        summary["n_trajectories"].clip(lower=1)
    )
    summary["ci95_low_um2"] = (
        summary["mean_msd_um2"] - FIT_CI_Z * summary["sem_msd_um2"]
    )
    summary["ci95_high_um2"] = (
        summary["mean_msd_um2"] + FIT_CI_Z * summary["sem_msd_um2"]
    )
    return summary.drop(columns=["std_msd_um2"])[columns]


def anomalous_diffusion_model(
    lag_time_s: np.ndarray,
    d_alpha: float,
    alpha: float,
) -> np.ndarray:
    return 4.0 * d_alpha * np.power(lag_time_s, alpha)


def fit_ensemble_msd(
    msd_summary: pd.DataFrame,
    state: str,
) -> dict:
    if msd_summary.empty:
        sub = pd.DataFrame(
            columns=[
                "lag_time_s",
                "mean_msd_um2",
                "n_trajectories",
            ]
        )
    else:
        sub = msd_summary[
            (msd_summary["state"] == state)
            & (msd_summary["lag_frame"] >= 1)
            & (msd_summary["lag_frame"] <= MAX_LAG_FRAMES)
        ].copy()
        for column in ["lag_time_s", "mean_msd_um2", "n_trajectories"]:
            sub[column] = pd.to_numeric(sub[column], errors="coerce")
        sub = sub[
            np.isfinite(sub["lag_time_s"].to_numpy(dtype=float))
            & np.isfinite(sub["mean_msd_um2"].to_numpy(dtype=float))
            & (sub["mean_msd_um2"].to_numpy(dtype=float) > 0)
        ]
    base = {
        "state": state,
        "model": "MSD(t) = 4 * D_alpha * t**alpha",
        "fit_weighting": FIT_WEIGHTING,
        "fit_initial_D_alpha": FIT_INITIAL_D_ALPHA,
        "fit_initial_alpha": FIT_INITIAL_ALPHA,
        "fit_D_alpha_lower_bound": FIT_D_ALPHA_BOUNDS[0],
        "fit_D_alpha_upper_bound": FIT_D_ALPHA_BOUNDS[1],
        "fit_alpha_lower_bound": FIT_ALPHA_BOUNDS[0],
        "fit_alpha_upper_bound": FIT_ALPHA_BOUNDS[1],
        "fit_min_lag_frame": 1,
        "fit_max_lag_frame": MAX_LAG_FRAMES,
        "fit_min_time_s": ANALYSIS_FRAME_INTERVAL_S,
        "fit_max_time_s": PLOT_MAX_TIME_S,
        "n_fit_points": int(len(sub)),
        "n_trajectories": (
            int(sub["n_trajectories"].max()) if not sub.empty else 0
        ),
    }
    if len(sub) < 3:
        return {
            **base,
            "fit_success": False,
            "fit_message": "Fewer than three finite ensemble MSD points.",
            "D_alpha": np.nan,
            "D_alpha_se": np.nan,
            "D_alpha_ci95_low": np.nan,
            "D_alpha_ci95_high": np.nan,
            "alpha": np.nan,
            "alpha_se": np.nan,
            "alpha_ci95_low": np.nan,
            "alpha_ci95_high": np.nan,
        }

    x = sub["lag_time_s"].to_numpy(dtype=float)
    y = sub["mean_msd_um2"].to_numpy(dtype=float)
    try:
        params, covariance = curve_fit(
            anomalous_diffusion_model,
            x,
            y,
            p0=[FIT_INITIAL_D_ALPHA, FIT_INITIAL_ALPHA],
            bounds=(
                [FIT_D_ALPHA_BOUNDS[0], FIT_ALPHA_BOUNDS[0]],
                [FIT_D_ALPHA_BOUNDS[1], FIT_ALPHA_BOUNDS[1]],
            ),
            maxfev=50000,
        )
        d_alpha, alpha = [float(value) for value in params]
        standard_errors = np.sqrt(np.diag(covariance))
        d_se, alpha_se = [float(value) for value in standard_errors]
        return {
            **base,
            "fit_success": True,
            "fit_message": "curve_fit converged; covariance-based 95% CI",
            "D_alpha": d_alpha,
            "D_alpha_se": d_se,
            "D_alpha_ci95_low": max(
                FIT_D_ALPHA_BOUNDS[0],
                d_alpha - FIT_CI_Z * d_se,
            ),
            "D_alpha_ci95_high": d_alpha + FIT_CI_Z * d_se,
            "alpha": alpha,
            "alpha_se": alpha_se,
            "alpha_ci95_low": max(
                FIT_ALPHA_BOUNDS[0],
                alpha - FIT_CI_Z * alpha_se,
            ),
            "alpha_ci95_high": min(
                FIT_ALPHA_BOUNDS[1],
                alpha + FIT_CI_Z * alpha_se,
            ),
        }
    except Exception as exc:
        return {
            **base,
            "fit_success": False,
            "fit_message": f"{type(exc).__name__}: {exc}",
            "D_alpha": np.nan,
            "D_alpha_se": np.nan,
            "D_alpha_ci95_low": np.nan,
            "D_alpha_ci95_high": np.nan,
            "alpha": np.nan,
            "alpha_se": np.nan,
            "alpha_ci95_low": np.nan,
            "alpha_ci95_high": np.nan,
        }


def build_activity_plot(
    msd_summary: pd.DataFrame,
    fit_results: pd.DataFrame,
):
    fig, axis = plt.subplots(figsize=(6.4, 4.8))
    colors = {"ON": "#e76f51", "OFF": "#4a90d9"}
    for state in ["ON", "OFF"]:
        sub = msd_summary[
            (msd_summary["state"] == state)
            & (msd_summary["lag_frame"] <= MAX_LAG_FRAMES)
        ].sort_values("lag_frame")
        if sub.empty:
            continue
        n_trajectories = int(sub["n_trajectories"].max())
        axis.plot(
            sub["lag_time_s"],
            sub["mean_msd_um2"],
            color=colors[state],
            linewidth=1.5,
            label=f"{ACTIVITY_DISPLAY_LABELS[state]} (n = {n_trajectories})",
        )
        axis.fill_between(
            sub["lag_time_s"].to_numpy(dtype=float),
            np.maximum(0.0, sub["ci95_low_um2"].to_numpy(dtype=float)),
            sub["ci95_high_um2"].to_numpy(dtype=float),
            color=colors[state],
            alpha=0.22,
        )
        fit_row = fit_results[fit_results["state"] == state]
        if not fit_row.empty and bool(fit_row.iloc[0]["fit_success"]):
            row = fit_row.iloc[0]
            x_fit = np.linspace(
                ANALYSIS_FRAME_INTERVAL_S,
                PLOT_MAX_TIME_S,
                300,
            )
            y_fit = anomalous_diffusion_model(
                x_fit,
                float(row["D_alpha"]),
                float(row["alpha"]),
            )
            axis.plot(
                x_fit,
                y_fit,
                color=colors[state],
                linestyle="--",
                linewidth=1.2,
                alpha=0.9,
            )
    axis.set_xlabel("lag time [s]")
    axis.set_ylabel(r"MSD ($\mu m^2$)")
    axis.set_xlim(0, PLOT_MAX_TIME_S)
    axis.set_ylim(0, PLOT_Y_MAX_UM2)
    axis.grid(alpha=0.25, linestyle=":")
    handles, labels = axis.get_legend_handles_labels()
    if handles:
        axis.legend(frameon=False, loc="upper left")
    fig.tight_layout()
    return fig


def save_activity_plot(
    msd_summary: pd.DataFrame,
    fit_results: pd.DataFrame,
    output_path: Path,
) -> Path:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig = build_activity_plot(msd_summary, fit_results)
    fig.savefig(output_path, dpi=200)
    plt.close(fig)
    return output_path


def save_qc_overlay(
    sample_id: str,
    mtetr_reference: np.ndarray,
    seed_xy: tuple[float, float],
    first_mtetr_xy: tuple[float, float],
    classification: dict,
    track: pd.DataFrame,
    output_path: Path,
) -> None:
    detected = track["used_for_msd"].fillna(False).astype(bool)
    x0, x1, y0, y1 = classification["roi_bounds"]
    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.5))

    axes[0].imshow(mtetr_reference, cmap="gray")
    axes[0].scatter(
        [seed_xy[0]],
        [seed_xy[1]],
        c="cyan",
        s=38,
        marker="+",
        label="mean-image seed",
    )
    axes[0].scatter(
        [first_mtetr_xy[0]],
        [first_mtetr_xy[1]],
        c="red",
        s=25,
        label="actual frame-0 mTetR",
    )
    axes[0].add_patch(
        plt.Rectangle(
            (x0, y0),
            x1 - x0,
            y1 - y0,
            fill=False,
            edgecolor="yellow",
            linewidth=1.2,
        )
    )
    axes[0].set_title("seed vs actual frame-0 coordinate")
    axes[0].legend(frameon=False, fontsize=7)
    axes[0].set_axis_off()

    axes[1].imshow(mtetr_reference, cmap="gray")
    if detected.any():
        axes[1].plot(
            track.loc[detected, "x_px"],
            track.loc[detected, "y_px"],
            color="yellow",
            linewidth=1.0,
        )
        axes[1].scatter(
            track.loc[detected, "x_px"],
            track.loc[detected, "y_px"],
            c=track.loc[detected, "frame"],
            cmap="viridis",
            s=10,
        )
    axes[1].set_title("trackpy-linked trajectory")
    axes[1].set_axis_off()

    axes[2].plot(
        track["time_s"],
        track["disp_um"],
        color="tab:red",
        linewidth=1.2,
        label="distance from actual frame 0",
    )
    axes[2].plot(
        track["time_s"],
        track["dx_um"],
        color="tab:blue",
        linewidth=0.9,
        alpha=0.8,
        label="dx",
    )
    axes[2].plot(
        track["time_s"],
        track["dy_um"],
        color="tab:orange",
        linewidth=0.9,
        alpha=0.8,
        label="dy",
    )
    axes[2].set_title(
        f"{sample_id} | activity="
        f"{ACTIVITY_DISPLAY_LABELS.get(classification['state'], classification['state'])} | "
        f"dt={ANALYSIS_FRAME_INTERVAL_S:.2f} s"
    )
    axes[2].set_xlabel("time [s]")
    axes[2].set_ylabel("displacement [um]")
    axes[2].grid(alpha=0.25, linestyle=":")
    axes[2].legend(frameon=False, fontsize=7)

    fig.tight_layout()
    fig.savefig(output_path, dpi=200)
    plt.close(fig)


def build_professor_style_csv(msd_per_trajectory: pd.DataFrame) -> pd.DataFrame:
    columns = ["lag time [s]", "msd", "sample", "state"]
    if msd_per_trajectory.empty:
        return pd.DataFrame(columns=columns)
    tidy = msd_per_trajectory.copy()
    numeric_sample = tidy["sample_id"].astype(str).str.extract(r"(\d+)")[0]
    tidy["sample"] = pd.to_numeric(numeric_sample, errors="coerce").fillna(0).astype(int)
    tidy = tidy.rename(
        columns={
            "lag_time_s": "lag time [s]",
            "msd_um2": "msd",
        }
    )
    return (
        tidy[columns]
        .sort_values(["state", "sample", "lag time [s]"])
        .reset_index(drop=True)
    )


print("MSD, anomalous-fit, and plotting functions ready.")


## 5. Reusable per-dataset analysis

This function performs the complete raw-to-figure workflow for one
biological replicate and writes to a new output-tagged folder.


In [ ]:
def discover_samples(data_dir: Path) -> list[dict]:
    auto_patterns = ("cell*.nd2", "cell*.tif", "cell*.tiff")
    movie_paths: list[Path] = []
    for pattern in auto_patterns:
        movie_paths.extend(sorted(data_dir.glob(pattern)))
    movie_paths = sorted({path.name: path for path in movie_paths}.values())
    if SAMPLE_IDS_FILTER:
        movie_paths = [
            path for path in movie_paths if path.stem in SAMPLE_IDS_FILTER
        ]
    if MAX_SAMPLES > 0:
        movie_paths = movie_paths[:MAX_SAMPLES]
    if not movie_paths:
        raise FileNotFoundError(
            f"No movies found in {data_dir} matching {auto_patterns}."
        )
    return [
        {
            "sample_id": path.stem,
            "movie_path": str(path),
            "pixel_size_um": None,
        }
        for path in movie_paths
    ]


def analyze_dataset(dataset: dict, output_tag: str = OUTPUT_TAG) -> dict:
    DATA_DIR = Path(dataset["data_dir"])
    OUTPUT_TAG = output_tag
    PROCESSED_DIR = (
        ALL_REPLICATES_OUTPUT_DIR
        / "per_dataset"
        / dataset["dataset_id"]
        / f"2_processed_{OUTPUT_TAG}"
    )
    TABLES_DIR = PROCESSED_DIR / "tables"
    PLOTS_DIR = PROCESSED_DIR / "plots"
    QC_DIR = PROCESSED_DIR / "QC"
    DEBUG_DIR = PROCESSED_DIR / "tracking_debug"
    CANDIDATE_DIR = PROCESSED_DIR / "candidates"
    CSV_EXPORT_DIR = ALL_REPLICATES_OUTPUT_DIR / "tidy_csv"

    for directory in [
        PROCESSED_DIR,
        TABLES_DIR,
        PLOTS_DIR,
        QC_DIR,
        DEBUG_DIR,
        CANDIDATE_DIR,
        CSV_EXPORT_DIR,
    ]:
        directory.mkdir(parents=True, exist_ok=True)

    SAMPLES = discover_samples(DATA_DIR)
    print("")
    print("=" * 72)
    print(
        f"{dataset['gene']} rep{dataset['replicate']} "
        f"({dataset['dataset_id']}): {len(SAMPLES)} movies"
    )
    print(f"Raw data: {DATA_DIR}")
    print(f"Output: {PROCESSED_DIR}")
    classification_rows: list[dict] = []
    tracking_tables: list[pd.DataFrame] = []
    candidate_tables: list[pd.DataFrame] = []
    msd_tables: list[pd.DataFrame] = []
    failure_rows: list[dict] = []

    print(f"Starting {OUTPUT_TAG} analysis for {len(SAMPLES)} sample(s).")

    for sample_index, sample in enumerate(SAMPLES, start=1):
        sample_id = str(sample["sample_id"])
        print(f"[{sample_index}/{len(SAMPLES)}] {sample_id}")
        try:
            mcp_raw, mtetr_raw, metadata, structure = load_movie_file(sample)
            pixel_size_um, pixel_source = resolve_pixel_size(sample, metadata)

            mcp_processed = preprocess_frame(mcp_raw)
            mtetr_processed = preprocess_movie(mtetr_raw)
            seed = detect_seed_from_mean(mtetr_processed, sample_id)
            seed_xy = (seed["seed_x_px"], seed["seed_y_px"])

            features, fixed_roi_bounds = detect_mtetr_candidates(
                mtetr_processed,
                seed_xy,
            )
            track, linked_features, selection = link_mtetr_candidates(
                features,
                seed_xy,
                n_frames=len(mtetr_processed),
                pixel_size_um=pixel_size_um,
            )
            first_mtetr_xy = (
                selection["first_mtetr_x_px"],
                selection["first_mtetr_y_px"],
            )
            classification = classify_mcp_state(
                mcp_image_raw=mcp_raw,
                mcp_image_processed=mcp_processed,
                first_mtetr_xy=first_mtetr_xy,
            )
            mtetr_ratio = mtetr_ratio_for_audit(mtetr_raw[0], first_mtetr_xy)
            qc = trajectory_qc(track)
            export_state = classification["state"] if qc["qc_pass"] else "EXCLUDE"

            classification_rows.append(
                {
                    "sample_id": sample_id,
                    "movie_path": sample["movie_path"],
                    "movie_structure": structure,
                    "state": export_state,
                    "raw_state": classification["state"],
                    "MCP_status": classification["MCP_status"],
                    "mcp_ratio_threshold": MCP_RATIO_THRESHOLD,
                    "mcp_max_distance_px": MCP_MAX_DISTANCE_PX,
                    "mcp_relative_intensity": classification["relative_intensity"],
                    "mcp_signal_intensity": classification["signal_intensity"],
                    "mcp_roi_mean_intensity": classification["roi_mean_intensity"],
                    "mcp_intensity_method": classification["intensity_method"],
                    "mcp_distance_px": classification["distance_px"],
                    "mcp_x_px": classification["mcp_x_px"],
                    "mcp_y_px": classification["mcp_y_px"],
                    "mtetr_mean_seed_x_px": seed["seed_x_px"],
                    "mtetr_mean_seed_y_px": seed["seed_y_px"],
                    "mtetr_seed_mode": seed["seed_mode"],
                    "mtetr_first_actual_frame_x_px": first_mtetr_xy[0],
                    "mtetr_first_actual_frame_y_px": first_mtetr_xy[1],
                    "mtetr_first_actual_frame_snr": selection["first_mtetr_snr"],
                    "mtetr_first_actual_frame_index": 0,
                    "mtetr_relative_intensity_audit": mtetr_ratio,
                    "selected_particle": selection["selected_particle"],
                    "pixel_size_um": pixel_size_um,
                    "pixel_size_source": pixel_source,
                    "metadata_mtetr_interval_s": metadata.get(
                        "metadata_mtetr_interval_s", np.nan
                    ),
                    "metadata_mcp_to_first_mtetr_gap_s": metadata.get(
                        "mcp_to_first_mtetr_gap_s", np.nan
                    ),
                    "analysis_frame_interval_s": ANALYSIS_FRAME_INTERVAL_S,
                    "analysis_time_axis_source": "publication_assumption_30ms",
                    "max_lag_frames": MAX_LAG_FRAMES,
                    "max_lag_time_s": PLOT_MAX_TIME_S,
                    "roi_size_px": 2 * ROI_HALF_SIZE_PX + 1,
                    "link_search_range_px": LINK_SEARCH_RANGE_PX,
                    "link_memory_frames": LINK_MEMORY_FRAMES,
                    "tracking_qc_pass": qc["qc_pass"],
                    "tracking_valid_frames": qc["valid_frames"],
                    "tracking_missing_fraction": qc["missing_fraction"],
                    "tracking_max_consecutive_detected_frames": qc[
                        "max_consecutive_detected_frames"
                    ],
                    "tracking_median_snr": qc["median_track_snr"],
                    "tracking_qc_rule": qc["qc_rule"],
                    "fixed_roi_x0": fixed_roi_bounds[0],
                    "fixed_roi_x1": fixed_roi_bounds[1],
                    "fixed_roi_y0": fixed_roi_bounds[2],
                    "fixed_roi_y1": fixed_roi_bounds[3],
                }
            )

            track.insert(0, "sample_id", sample_id)
            track.insert(1, "state", export_state)
            track.insert(2, "raw_state", classification["state"])
            track.insert(3, "tracking_qc_pass", bool(qc["qc_pass"]))
            track.insert(4, "pixel_size_um", pixel_size_um)
            track.insert(5, "analysis_frame_interval_s", ANALYSIS_FRAME_INTERVAL_S)
            track.insert(6, "link_search_range_px", LINK_SEARCH_RANGE_PX)
            track.insert(7, "link_memory_frames", LINK_MEMORY_FRAMES)
            tracking_tables.append(track)

            linked_features.insert(0, "sample_id", sample_id)
            linked_features.insert(
                1,
                "selected_particle",
                selection["selected_particle"],
            )
            candidate_tables.append(linked_features)
            linked_features.to_csv(
                CANDIDATE_DIR / f"{sample_id}_{OUTPUT_TAG}_linked_candidates.csv",
                index=False,
            )
            track.to_csv(
                DEBUG_DIR / f"{sample_id}_{OUTPUT_TAG}_tracking_debug.csv",
                index=False,
            )

            save_qc_overlay(
                sample_id=sample_id,
                mtetr_reference=np.mean(
                    mtetr_processed[: min(10, len(mtetr_processed))],
                    axis=0,
                ),
                seed_xy=seed_xy,
                first_mtetr_xy=first_mtetr_xy,
                classification=classification,
                track=track,
                output_path=QC_DIR / f"{sample_id}_{OUTPUT_TAG}_qc.png",
            )

            if qc["qc_pass"] and classification["state"] in {"ON", "OFF"}:
                msd_single = compute_time_averaged_msd(
                    track,
                    sample_id=sample_id,
                    state=classification["state"],
                )
                if not msd_single.empty:
                    msd_tables.append(msd_single)

            print(
                "  "
                f"state={classification['state']}, "
                f"ratio={classification['relative_intensity']:.3f}, "
                f"distance={classification['distance_px']:.2f} px, "
                f"max_consecutive={qc['max_consecutive_detected_frames']}, "
                f"qc={qc['qc_pass']}"
            )
        except Exception as exc:
            message = f"{type(exc).__name__}: {exc}"
            traceback_text = traceback.format_exc()
            failure_rows.append(
                {
                    "sample_id": sample_id,
                    "movie_path": sample["movie_path"],
                    "error": message,
                    "traceback": traceback_text,
                }
            )
            classification_rows.append(
                {
                    "sample_id": sample_id,
                    "movie_path": sample["movie_path"],
                    "state": "EXCLUDE",
                    "raw_state": "ERROR",
                    "tracking_qc_pass": False,
                    "error": message,
                    "traceback": traceback_text,
                    "analysis_frame_interval_s": ANALYSIS_FRAME_INTERVAL_S,
                    "max_lag_frames": MAX_LAG_FRAMES,
                    "max_lag_time_s": PLOT_MAX_TIME_S,
                    "link_search_range_px": LINK_SEARCH_RANGE_PX,
                    "link_memory_frames": LINK_MEMORY_FRAMES,
                }
            )
            print(f"  FAILED: {message}")

    classification_df = pd.DataFrame(classification_rows)
    tracking_df = (
        pd.concat(tracking_tables, ignore_index=True)
        if tracking_tables
        else pd.DataFrame()
    )
    candidate_df = (
        pd.concat(candidate_tables, ignore_index=True)
        if candidate_tables
        else pd.DataFrame()
    )
    msd_per_trajectory_df = (
        pd.concat(msd_tables, ignore_index=True)
        if msd_tables
        else pd.DataFrame()
    )
    failure_df = pd.DataFrame(failure_rows)
    msd_summary_df = summarize_ensemble_msd(msd_per_trajectory_df)
    fit_results_df = pd.DataFrame(
        [fit_ensemble_msd(msd_summary_df, state) for state in ["ON", "OFF"]]
    )
    professor_tidy_df = build_professor_style_csv(msd_per_trajectory_df)

    for table in [
        classification_df,
        tracking_df,
        candidate_df,
        msd_per_trajectory_df,
        msd_summary_df,
        fit_results_df,
        professor_tidy_df,
        failure_df,
    ]:
        table.insert(0, "replicate", int(dataset["replicate"]))
        table.insert(0, "gene", dataset["gene"])
        table.insert(0, "dataset_id", dataset["dataset_id"])

    classification_path = TABLES_DIR / f"{OUTPUT_TAG}_classification.csv"
    tracking_path = TABLES_DIR / f"{OUTPUT_TAG}_tracking_coordinates.csv"
    candidate_path = TABLES_DIR / f"{OUTPUT_TAG}_linked_candidates.csv"
    failure_path = TABLES_DIR / f"{OUTPUT_TAG}_failures.csv"
    msd_path = TABLES_DIR / f"{OUTPUT_TAG}_msd_per_trajectory.csv"
    msd_summary_path = TABLES_DIR / f"{OUTPUT_TAG}_msd_summary.csv"
    fit_path = TABLES_DIR / f"{OUTPUT_TAG}_anomalous_fit.csv"
    plot_path = PLOTS_DIR / f"{OUTPUT_TAG}_Active_vs_Inactive_MSD_0to1p98s.png"
    professor_csv_path = (
        CSV_EXPORT_DIR / f"{DATA_DIR.parent.name}_{OUTPUT_TAG}_tidy.csv"
    )

    classification_df.to_csv(classification_path, index=False)
    tracking_df.to_csv(tracking_path, index=False)
    candidate_df.to_csv(candidate_path, index=False)
    failure_df.to_csv(failure_path, index=False)
    msd_per_trajectory_df.to_csv(msd_path, index=False)
    msd_summary_df.to_csv(msd_summary_path, index=False)
    fit_results_df.to_csv(fit_path, index=False)
    professor_tidy_df.to_csv(professor_csv_path, index=False)

    if not msd_summary_df.empty:
        save_activity_plot(msd_summary_df, fit_results_df, plot_path)

    run_configuration = {
        "dataset_id": dataset["dataset_id"],
        "gene": dataset["gene"],
        "replicate": int(dataset["replicate"]),
        "data_dir": str(DATA_DIR),
        "output_tag": OUTPUT_TAG,
        "n_input_samples": len(SAMPLES),
        "sample_ids_filter": sorted(SAMPLE_IDS_FILTER),
        "n_successfully_tracked": len(tracking_tables),
        "n_failures": len(failure_df),
        "mcp_ratio_threshold": MCP_RATIO_THRESHOLD,
        "mcp_max_distance_px": MCP_MAX_DISTANCE_PX,
        "roi_size_px": 2 * ROI_HALF_SIZE_PX + 1,
        "link_search_range_px": LINK_SEARCH_RANGE_PX,
        "link_memory_frames": LINK_MEMORY_FRAMES,
        "minimum_consecutive_detected_frames": MIN_CONSECUTIVE_DETECTED_FRAMES,
        "analysis_frame_interval_s": ANALYSIS_FRAME_INTERVAL_S,
        "max_lag_frames": MAX_LAG_FRAMES,
        "max_lag_time_s": PLOT_MAX_TIME_S,
        "fit_model": "MSD(t) = 4 * D_alpha * t**alpha",
        "fit_weighting": FIT_WEIGHTING,
        "fit_initial_values": {
            "D_alpha": FIT_INITIAL_D_ALPHA,
            "alpha": FIT_INITIAL_ALPHA,
        },
        "fit_bounds": {
            "D_alpha": [FIT_D_ALPHA_BOUNDS[0], "inf"],
            "alpha": list(FIT_ALPHA_BOUNDS),
        },
        "python": sys.version,
        "platform": platform.platform(),
        "packages": {
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "scipy": scipy.__version__,
            "scikit_image": skimage.__version__,
            "matplotlib": matplotlib.__version__,
            "trackpy": tp.__version__,
            "nd2": nd2.__version__,
            "tifffile": tifffile.__version__,
        },
    }
    with (PROCESSED_DIR / "run_configuration.json").open("w", encoding="utf-8") as handle:
        json.dump(run_configuration, handle, ensure_ascii=False, indent=2)

    if tracking_df.empty:
        raise RuntimeError(
            "No sample completed tracking. Inspect the failures table: "
            f"{failure_path}"
        )

    print("")
    print("Batch analysis complete.")
    print(f"classification: {classification_path}")
    print(f"tracking: {tracking_path}")
    print(f"linked candidates: {candidate_path}")
    print(f"failures: {failure_path}")
    print(f"MSD per trajectory: {msd_path}")
    print(f"ensemble MSD: {msd_summary_path}")
    print(f"anomalous fit: {fit_path}")
    print(f"tidy CSV: {professor_csv_path}")
    print(f"plot: {plot_path if plot_path.exists() else 'not generated'}")

    display(activity_display_table(classification_df.head()))
    display(activity_display_table(fit_results_df))
    return {
        "dataset_id": dataset["dataset_id"],
        "gene": dataset["gene"],
        "replicate": int(dataset["replicate"]),
        "data_dir": DATA_DIR,
        "processed_dir": PROCESSED_DIR,
        "classification": classification_df,
        "tracking": tracking_df,
        "linked_candidates": candidate_df,
        "msd_per_trajectory": msd_per_trajectory_df,
        "msd_summary": msd_summary_df,
        "fit_results": fit_results_df,
        "failures": failure_df,
        "plot_path": plot_path,
        "run_configuration": run_configuration,
    }


## 6. Run all four biological-replicate datasets

The four datasets are processed sequentially to avoid concurrent
writes and excessive memory use. Per-movie failures are recorded
without stopping the remaining movies.


In [ ]:
RESULTS = []
for dataset_config in DATASET_CONFIGS:
    RESULTS.append(analyze_dataset(dataset_config))

print("")
print(f"Completed datasets: {len(RESULTS)}")


## 7. Consolidate replicates and create final figures

The first figure keeps biological replicates separate in a 2 x 2
layout. The second figure pools cell trajectories across the two
replicates for descriptive visualization only; it is not a
replicate-level inferential analysis.


In [ ]:
def draw_activity_axis(
    axis,
    msd_summary: pd.DataFrame,
    fit_results: pd.DataFrame,
    title: str,
) -> None:
    colors = {"ON": "#e76f51", "OFF": "#4a90d9"}
    for state in ["ON", "OFF"]:
        subset = msd_summary[
            (msd_summary["state"] == state)
            & (msd_summary["lag_frame"] <= MAX_LAG_FRAMES)
        ].sort_values("lag_frame")
        if subset.empty:
            continue
        n_trajectories = int(subset["n_trajectories"].max())
        axis.plot(
            subset["lag_time_s"],
            subset["mean_msd_um2"],
            color=colors[state],
            linewidth=1.5,
            label=(
                f"{ACTIVITY_DISPLAY_LABELS[state]} "
                f"(n = {n_trajectories})"
            ),
        )
        axis.fill_between(
            subset["lag_time_s"].to_numpy(dtype=float),
            np.maximum(
                0.0,
                subset["ci95_low_um2"].to_numpy(dtype=float),
            ),
            subset["ci95_high_um2"].to_numpy(dtype=float),
            color=colors[state],
            alpha=0.22,
        )
        fit_row = fit_results[fit_results["state"] == state]
        if not fit_row.empty and bool(fit_row.iloc[0]["fit_success"]):
            row = fit_row.iloc[0]
            x_fit = np.linspace(
                ANALYSIS_FRAME_INTERVAL_S,
                PLOT_MAX_TIME_S,
                300,
            )
            y_fit = anomalous_diffusion_model(
                x_fit,
                float(row["D_alpha"]),
                float(row["alpha"]),
            )
            axis.plot(
                x_fit,
                y_fit,
                color=colors[state],
                linestyle="--",
                linewidth=1.2,
                alpha=0.9,
            )
    axis.set_title(title, fontstyle="italic" if title.split()[0] != "Dnmt3L" else "normal")
    axis.set_xlabel("lag time [s]")
    axis.set_ylabel(r"MSD ($\mu m^2$)")
    axis.set_xlim(0, PLOT_MAX_TIME_S)
    axis.set_ylim(0, PLOT_Y_MAX_UM2)
    axis.grid(alpha=0.25, linestyle=":")
    handles, labels = axis.get_legend_handles_labels()
    if handles:
        axis.legend(frameon=False, loc="upper left", fontsize=8)


all_classification_df = pd.concat(
    [result["classification"] for result in RESULTS],
    ignore_index=True,
)
all_fit_results_df = pd.concat(
    [result["fit_results"] for result in RESULTS],
    ignore_index=True,
)
all_failures_df = pd.concat(
    [result["failures"] for result in RESULTS],
    ignore_index=True,
)

combined_msd_parts = []
for result in RESULTS:
    part = result["msd_per_trajectory"].copy()
    if part.empty:
        continue
    part["source_sample_id"] = part["sample_id"].astype(str)
    part["sample_id"] = (
        result["dataset_id"] + "::" + part["source_sample_id"]
    )
    combined_msd_parts.append(part)
if combined_msd_parts:
    all_msd_per_trajectory_df = pd.concat(
        combined_msd_parts,
        ignore_index=True,
    )
else:
    all_msd_per_trajectory_df = pd.DataFrame(
        columns=[
            "dataset_id",
            "gene",
            "replicate",
            "sample_id",
            "source_sample_id",
            "state",
            "lag_frame",
            "lag_time_s",
            "msd_um2",
            "n_pairs",
        ]
    )

count_summary_df = (
    all_classification_df.groupby(
        ["gene", "replicate", "dataset_id", "state"],
        dropna=False,
    )
    .size()
    .rename("n_movies")
    .reset_index()
)

replicate_summary_df = pd.concat(
    [result["msd_summary"] for result in RESULTS],
    ignore_index=True,
)

gene_pooled_summary_tables = []
gene_pooled_fit_rows = []
for gene in ["Usp5", "Dnmt3L"]:
    gene_msd = all_msd_per_trajectory_df[
        all_msd_per_trajectory_df["gene"] == gene
    ].copy()
    gene_summary = summarize_ensemble_msd(gene_msd)
    gene_summary.insert(0, "gene", gene)
    gene_summary.insert(1, "pooling", "cell_level_across_two_replicates")
    gene_pooled_summary_tables.append(gene_summary)
    for state in ["ON", "OFF"]:
        fit_row = fit_ensemble_msd(gene_summary, state)
        fit_row["gene"] = gene
        fit_row["pooling"] = "cell_level_across_two_replicates"
        gene_pooled_fit_rows.append(fit_row)

gene_pooled_summary_df = pd.concat(
    gene_pooled_summary_tables,
    ignore_index=True,
)
gene_pooled_fit_df = pd.DataFrame(gene_pooled_fit_rows)

all_classification_path = (
    ALL_REPLICATES_OUTPUT_DIR / "all_replicates_classification.csv"
)
all_msd_path = (
    ALL_REPLICATES_OUTPUT_DIR / "all_replicates_msd_per_trajectory.csv"
)
replicate_summary_path = (
    ALL_REPLICATES_OUTPUT_DIR / "replicate_msd_summary.csv"
)
replicate_fit_path = (
    ALL_REPLICATES_OUTPUT_DIR / "replicate_anomalous_fit.csv"
)
pooled_summary_path = (
    ALL_REPLICATES_OUTPUT_DIR / "gene_pooled_msd_summary.csv"
)
pooled_fit_path = (
    ALL_REPLICATES_OUTPUT_DIR / "gene_pooled_anomalous_fit.csv"
)
count_summary_path = (
    ALL_REPLICATES_OUTPUT_DIR / "dataset_state_counts.csv"
)
failure_summary_path = (
    ALL_REPLICATES_OUTPUT_DIR / "all_replicates_failures.csv"
)

all_classification_df.to_csv(all_classification_path, index=False)
all_msd_per_trajectory_df.to_csv(all_msd_path, index=False)
replicate_summary_df.to_csv(replicate_summary_path, index=False)
all_fit_results_df.to_csv(replicate_fit_path, index=False)
gene_pooled_summary_df.to_csv(pooled_summary_path, index=False)
gene_pooled_fit_df.to_csv(pooled_fit_path, index=False)
count_summary_df.to_csv(count_summary_path, index=False)
all_failures_df.to_csv(failure_summary_path, index=False)

result_by_design = {
    (result["gene"], result["replicate"]): result
    for result in RESULTS
}
genes = ["Usp5", "Dnmt3L"]

replicate_figure, replicate_axes = plt.subplots(
    2,
    2,
    figsize=(9.6, 8.2),
    sharex=True,
    sharey=True,
)
for row, replicate in enumerate([1, 2]):
    for column, gene in enumerate(genes):
        result = result_by_design[(gene, replicate)]
        draw_activity_axis(
            replicate_axes[row, column],
            result["msd_summary"],
            result["fit_results"],
            f"{gene} rep{replicate}",
        )
replicate_figure.tight_layout()
replicate_figure_path = (
    ALL_REPLICATES_OUTPUT_DIR
    / "MSD_Active_Inactive_by_biological_replicate.png"
)
replicate_figure_pdf_path = replicate_figure_path.with_suffix(".pdf")
replicate_figure.savefig(replicate_figure_path, dpi=200)
replicate_figure.savefig(replicate_figure_pdf_path)
plt.close(replicate_figure)

pooled_figure, pooled_axes = plt.subplots(
    1,
    2,
    figsize=(9.6, 4.2),
    sharex=True,
    sharey=True,
)
for column, gene in enumerate(genes):
    draw_activity_axis(
        pooled_axes[column],
        gene_pooled_summary_df[
            gene_pooled_summary_df["gene"] == gene
        ],
        gene_pooled_fit_df[gene_pooled_fit_df["gene"] == gene],
        gene,
    )
pooled_figure.suptitle(
    "Descriptive cell-level pooling across two biological replicates",
    fontsize=11,
)
pooled_figure.tight_layout()
pooled_figure_path = (
    ALL_REPLICATES_OUTPUT_DIR
    / "MSD_Active_Inactive_pooled_by_gene.png"
)
pooled_figure_pdf_path = pooled_figure_path.with_suffix(".pdf")
pooled_figure.savefig(pooled_figure_path, dpi=200)
pooled_figure.savefig(pooled_figure_pdf_path)
plt.close(pooled_figure)

print("Combined tables and Active/Inactive figures saved:")
for path in [
    all_classification_path,
    all_msd_path,
    replicate_summary_path,
    replicate_fit_path,
    pooled_summary_path,
    pooled_fit_path,
    count_summary_path,
    failure_summary_path,
    replicate_figure_path,
    replicate_figure_pdf_path,
    pooled_figure_path,
    pooled_figure_pdf_path,
]:
    print(path)

display(activity_display_table(count_summary_df))
display(activity_display_table(all_fit_results_df))


## 8. Validation


In [ ]:
assert len(RESULTS) == 4
assert {
    (result["gene"], result["replicate"])
    for result in RESULTS
} == expected_design

validation_rows = []
is_full_run = MAX_SAMPLES == 0 and not SAMPLE_IDS_FILTER
for result in RESULTS:
    raw_count = len(list(result["data_dir"].glob("cell*.nd2")))
    expected_count = (
        raw_count if is_full_run else len(discover_samples(result["data_dir"]))
    )
    classification = result["classification"]
    tracking = result["tracking"]
    msd = result["msd_per_trajectory"]
    fit = result["fit_results"]
    checks = {
        "classification_rows_match_expected": (
            len(classification) == expected_count
        ),
        "frame0_present": bool(
            (not is_full_run and tracking.empty)
            or (
                not tracking.empty
                and (tracking["frame"] == 0).any()
            )
        ),
        "search_range_is_2": bool(
            np.allclose(
                classification["link_search_range_px"]
                .dropna()
                .to_numpy(dtype=float),
                LINK_SEARCH_RANGE_PX,
            )
        ),
        "memory_is_4": bool(
            (
                classification["link_memory_frames"]
                .dropna()
                .astype(int)
                == LINK_MEMORY_FRAMES
            ).all()
        ),
        "analysis_dt_is_0p03": bool(
            np.allclose(
                classification["analysis_frame_interval_s"]
                .dropna()
                .to_numpy(dtype=float),
                ANALYSIS_FRAME_INTERVAL_S,
            )
        ),
        "max_lag_is_66": bool(
            (not is_full_run and msd.empty)
            or (
                not msd.empty
                and int(msd["lag_frame"].max()) == MAX_LAG_FRAMES
            )
        ),
        "max_time_is_1p98": bool(
            (not is_full_run and msd.empty)
            or (
                not msd.empty
                and np.isclose(float(msd["lag_time_s"].max()), 1.98)
            )
        ),
        "active_inactive_fit_present": (
            set(fit["state"]) == {"ON", "OFF"}
        ),
        "replicate_plot_exists": bool(
            (not is_full_run and msd.empty)
            or (
                result["plot_path"].exists()
                and result["plot_path"].stat().st_size > 0
            )
        ),
    }
    validation_rows.append(
        {
            "dataset_id": result["dataset_id"],
            "gene": result["gene"],
            "replicate": result["replicate"],
            "raw_nd2": raw_count,
            "msd_trajectories": (
                int(msd["sample_id"].nunique()) if not msd.empty else 0
            ),
            "all_checks_pass": all(checks.values()),
            **checks,
        }
    )

validation_df = pd.DataFrame(validation_rows)
validation_path = (
    ALL_REPLICATES_OUTPUT_DIR / "all_replicates_validation.csv"
)
validation_df.to_csv(validation_path, index=False)

assert bool(validation_df["all_checks_pass"].all())
assert replicate_figure_path.exists()
assert pooled_figure_path.exists()
assert replicate_figure_path.stat().st_size > 0
assert pooled_figure_path.stat().st_size > 0
assert set(gene_pooled_fit_df["gene"]) == {"Usp5", "Dnmt3L"}
assert set(gene_pooled_fit_df["state"]) == {"ON", "OFF"}

print("All four datasets passed validation.")
print(f"Validation table: {validation_path}")
display(validation_df)


In [ ]:
# Cell 1: load all required data and packages without rerunning the analysis
from pathlib import Path
import os

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
from IPython.display import display

OUTPUT_DIR = Path(
    os.environ.get(
        "MSD_ALL_REPLICATES_OUTPUT_DIR",
        str(GAO2026_OUTPUT_ROOT / '07_MSD'),
    )
)

MSD_SUMMARY_PATH = OUTPUT_DIR / "replicate_msd_summary.csv"
MSD_FIT_PATH = OUTPUT_DIR / "replicate_anomalous_fit.csv"

for required_path in [MSD_SUMMARY_PATH, MSD_FIT_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

msd_summary = pd.read_csv(MSD_SUMMARY_PATH)
msd_fits = pd.read_csv(MSD_FIT_PATH)

required_summary_columns = {
    "gene",
    "replicate",
    "state",
    "lag_frame",
    "lag_time_s",
    "n_trajectories",
    "mean_msd_um2",
    "ci95_low_um2",
    "ci95_high_um2",
}
required_fit_columns = {
    "gene",
    "replicate",
    "state",
    "fit_success",
    "D_alpha",
    "alpha",
}

missing_summary_columns = required_summary_columns - set(msd_summary.columns)
missing_fit_columns = required_fit_columns - set(msd_fits.columns)

if missing_summary_columns:
    raise ValueError(
        f"Missing columns in {MSD_SUMMARY_PATH.name}: "
        f"{sorted(missing_summary_columns)}"
    )
if missing_fit_columns:
    raise ValueError(
        f"Missing columns in {MSD_FIT_PATH.name}: "
        f"{sorted(missing_fit_columns)}"
    )

PLOT_GENES = ["Usp5", "Dnmt3L"]
PLOT_REPLICATES = [1, 2]
PANEL_LABELS = ["a", "b", "c", "d"]

STATE_ORDER = ["ON", "OFF"]
STATE_LABELS = {
    "ON": "Active",
    "OFF": "Inactive",
}
STATE_COLORS = {
    "ON": "#D55E00",
    "OFF": "#0072B2",
}

FONT_NAME = "Arial"

# Register Arial explicitly when it is available at a common macOS/Linux path.
arial_font_paths = [
    Path("/System/Library/Fonts/Supplemental/Arial.ttf"),
    Path("/System/Library/Fonts/Supplemental/Arial Bold.ttf"),
    Path("/System/Library/Fonts/Supplemental/Arial Italic.ttf"),
    Path("/System/Library/Fonts/Supplemental/Arial Bold Italic.ttf"),
    Path("/usr/share/fonts/truetype/msttcorefonts/Arial.ttf"),
    Path("/usr/share/fonts/truetype/msttcorefonts/Arial_Bold.ttf"),
    Path("/usr/share/fonts/truetype/msttcorefonts/Arial_Italic.ttf"),
    Path("/usr/share/fonts/truetype/msttcorefonts/Arial_Bold_Italic.ttf"),
    Path.home() / ".fonts/Arial.ttf",
]

for font_path in arial_font_paths:
    if font_path.exists():
        font_manager.fontManager.addfont(str(font_path))

available_font_names = {
    font_entry.name for font_entry in font_manager.fontManager.ttflist
}
if FONT_NAME not in available_font_names:
    raise RuntimeError(
        "Arial is not available in this Python environment. "
        "Install or register Arial before generating the final PDF."
    )

OUTPUT_PDF = (
    OUTPUT_DIR
    / "MSD_Active_Inactive_by_biological_replicate_"
    "publication_Arial.pdf"
)

print(f"MSD summary: {MSD_SUMMARY_PATH}")
print(f"Fit results: {MSD_FIT_PATH}")
print(f"Output PDF: {OUTPUT_PDF}")
print(f"Font: {FONT_NAME}")

In [ ]:
# Cell 2: generate and export the publication-style PDF
def fit_succeeded(value) -> bool:
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() == "true"


FONT_size = 10
PANEL_LABELS = ["d", "e", "f", "g"]

def draw_msd_panel(
    axis: plt.Axes,
    gene: str,
    replicate: int,
) -> None:
    panel_summary = msd_summary[
        (msd_summary["gene"] == gene)
        & (msd_summary["replicate"] == replicate)
    ].copy()

    panel_fits = msd_fits[
        (msd_fits["gene"] == gene)
        & (msd_fits["replicate"] == replicate)
    ].copy()

    if panel_summary.empty:
        raise ValueError(
            f"No MSD summary was found for {gene}, replicate {replicate}."
        )

    for state in STATE_ORDER:
        state_summary = panel_summary[
            panel_summary["state"] == state
        ].sort_values("lag_frame")

        if state_summary.empty:
            continue

        x = state_summary["lag_time_s"].to_numpy(dtype=float)
        mean_msd = state_summary["mean_msd_um2"].to_numpy(dtype=float)
        ci_low = np.maximum(
            0.0,
            state_summary["ci95_low_um2"].to_numpy(dtype=float),
        )
        ci_high = state_summary["ci95_high_um2"].to_numpy(dtype=float)
        n_trajectories = int(state_summary["n_trajectories"].max())

        axis.fill_between(
            x,
            ci_low,
            ci_high,
            color=STATE_COLORS[state],
            alpha=0.18,
            linewidth=0,
        )
        axis.plot(
            x,
            mean_msd,
            color=STATE_COLORS[state],
            linewidth=1.5,
            label=f"{STATE_LABELS[state]} (n = {n_trajectories})",
        )

        fit_row = panel_fits[panel_fits["state"] == state]
        if (
            not fit_row.empty
            and fit_succeeded(fit_row.iloc[0]["fit_success"])
        ):
            fit_row = fit_row.iloc[0]
            x_fit = np.linspace(
                max(0.001, float(x.min())),
                float(x.max()),
                300,
            )
            y_fit = (
                4.0
                * float(fit_row["D_alpha"])
                * np.power(x_fit, float(fit_row["alpha"]))
            )

            axis.plot(
                x_fit,
                y_fit,
                color=STATE_COLORS[state],
                linewidth=0.9,
                linestyle=(0, (3, 2)),
            )

    # Separate text objects keep only the gene name italicized.
    axis.text(
        0.5,
        1.075,
        gene,
        transform=axis.transAxes,
        ha="center",
        va="bottom",
        fontsize=FONT_size,
        fontfamily=FONT_NAME,
        fontstyle="italic",
    )
    axis.text(
        0.5,
        1.015,
        f"Replicate {replicate}",
        transform=axis.transAxes,
        ha="center",
        va="bottom",
        fontsize=FONT_size,
        fontfamily=FONT_NAME,
    )

    axis.set_xlim(0.0, 2.0)
    axis.set_ylim(0.0, 0.04)
    axis.set_xticks(np.arange(0.0, 2.01, 0.5))
    axis.set_yticks(np.arange(0.0, 0.0401, 0.01))

    axis.tick_params(
        axis="both",
        which="major",
        direction="out",
        length=3,
        width=0.7,
        pad=2,
        labelsize=FONT_size,
    )

    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.spines["left"].set_linewidth(0.8)
    axis.spines["bottom"].set_linewidth(0.8)

    axis.legend(
        loc="upper left",
        frameon=False,
        fontsize=FONT_size,
        handlelength=2.4,
        borderaxespad=0.2,
        labelspacing=0.25,
    )



publication_rc = {
    "font.family": FONT_NAME,
    "font.sans-serif": [FONT_NAME],
    "font.size": FONT_size,
    "axes.labelsize": FONT_size,
    "axes.titlesize": FONT_size,
    "axes.linewidth": 0.8,
    "xtick.labelsize": FONT_size,
    "ytick.labelsize": FONT_size,
    "legend.fontsize": FONT_size,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
}

with mpl.rc_context(publication_rc):
    figure, axes = plt.subplots(
        2,
        2,
        figsize=(180 / 25.4, 142 / 25.4),
        sharex=True,
        sharey=True,
    )

    panel_index = 0
    for row_index, replicate in enumerate(PLOT_REPLICATES):
        for column_index, gene in enumerate(PLOT_GENES):
            axis = axes[row_index, column_index]

            draw_msd_panel(
                axis=axis,
                gene=gene,
                replicate=replicate,
            )

            axis.text(
                -0.17,
                1.09,
                PANEL_LABELS[panel_index],
                transform=axis.transAxes,
                ha="left",
                va="bottom",
                fontsize=FONT_size * 1.33,
                fontfamily=FONT_NAME,
                fontweight="bold",
            )
            panel_index += 1

    for axis in axes[-1, :]:
        axis.set_xlabel("Time lag (s)")

    for axis in axes[:, 0]:
        axis.set_ylabel(r"MSD ($\mathrm{\mu m}^2$)")

    figure.subplots_adjust(
        left=0.105,
        right=0.985,
        bottom=0.105,
        top=0.925,
        wspace=0.20,
        hspace=0.32,
    )

    figure.savefig(
        OUTPUT_PDF,
        format="pdf",
        bbox_inches="tight",
        pad_inches=0.03,
        facecolor="white",
    )

    display(figure)
    plt.close(figure)

if not OUTPUT_PDF.exists() or OUTPUT_PDF.stat().st_size == 0:
    raise RuntimeError(f"PDF export failed: {OUTPUT_PDF}")

print(f"Saved: {OUTPUT_PDF}")